# 인공지능모델운영 과제: Cross Validation & Hyperparameter Tuning (Optuna)

```
인공지능모델운영 과제: Cross Validation & Hyperparameter Tuning (Optuna)
Titanic 생존 예측: 6개 모델 Baseline vs. Optuna Tuned 비교

실행 흐름
  1) 데이터 로드 및 무상태(stateless) 정제
  2) 전처리 Pipeline 구성: 결측치 대체, 인코딩, 스케일링을 fold 내부에서 fit하여 Data Leakage 차단
  3) Baseline: 기본 하이퍼파라미터로 Stratified 5-Fold 평가
  4) Optuna(TPE Sampler, 모델당 50 trials, F1 최대화) 튜닝
  5) Tuned: 최적 파라미터로 동일한 5-Fold 재평가 및 Baseline 대비 변화량 출력

Colab 사용법: 아래 '# %%' 단위로 셀을 나누어 붙여 넣거나, 파일 전체를 한 셀에서 실행.
```

### [Cell 1] 라이브러리 설치 (Colab에서는 주석을 해제하고 실행)

In [ ]:
!pip install -q optuna xgboost scikit-learn pandas numpy

### [Cell 2] Import 및 재현성 설정

In [ ]:
import os
import random
import warnings
from typing import Callable

import numpy as np
import pandas as pd
import optuna
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

try:  # Colab/Jupyter에서는 스타일이 적용된 표로, 그 외에는 텍스트로 출력
    from IPython.display import display
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42


def set_seed(seed: int = RANDOM_STATE) -> None:
    """Python, NumPy, 해시 시드를 고정 (모델별 random_state, Optuna sampler seed와 함께 재현성 보장)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


set_seed()

### [Cell 3] 실험 설정

In [ ]:
TRAIN_URL = (
    "https://raw.githubusercontent.com/wesm/pydata-book/"
    "3rd-edition/datasets/titanic/train.csv"
)
N_SPLITS = 5          # Stratified K-Fold 분할 수
N_TRIALS = 50         # 모델별 Optuna trial 수
TUNING_METRIC = "f1"  # Optuna 목적 함수 (평균 CV F1-Score 최대화)

# 호칭(Title) -> 5개 범주 매핑. 매핑되지 않는 희소 호칭은 'Rare'로 처리
TITLE_MAPPING = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    **{t: "Rare" for t in [
        "Lady", "Countess", "Capt", "Col", "Don", "Dr",
        "Major", "Rev", "Sir", "Jonkheer", "Dona",
    ]},
}

NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]   # StandardScaler 대상
CAT_COLS = ["Sex", "Embarked", "Title"]        # OneHotEncoder(drop='first') 대상
PASS_COLS = ["Pclass"]                         # 서열형 변수: 그대로 통과

# 교수님 요청 표 형식
METRIC_COLS = ["FN+FP", "FPR", "Recall", "Precision", "F1-Score", "PR_AUC", "AUROC"]
LOWER_IS_BETTER = {"FN+FP", "FPR"}

### [Cell 4] 데이터 로드 및 무상태 정제

In [ ]:
def load_and_clean(url: str = TRAIN_URL) -> tuple[pd.DataFrame, pd.Series]:
    """
    통계량(평균, 중앙값 등)을 사용하지 않는 변환만 수행.
    -> fold 분할 이전에 적용해도 Data Leakage가 발생하지 않음.
    """
    df = pd.read_csv(url)
    df["Title"] = (
        df["Name"]
        .str.extract(r" ([A-Za-z]+)\.", expand=False)  # 'Braund, Mr. Owen' -> 'Mr'
        .map(TITLE_MAPPING)
        .fillna("Rare")
    )
    df = df.drop(columns=["PassengerId", "Ticket", "Cabin", "Name"])
    return df.drop(columns="Survived"), df["Survived"]


def summarize(df: pd.DataFrame, n_samples: int = 5) -> pd.DataFrame:
    """컬럼별 dtype, 고유값 수, 결측치 수/비율, 샘플 값 요약."""
    return pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.values,
        "N_Unique": df.nunique().values,
        "Null_Count": df.isnull().sum().values,
        "Null_Ratio(%)": (df.isnull().mean() * 100).round(2).values,
        "Samples": [list(df[c].dropna().unique()[:n_samples]) for c in df.columns],
    })

### [Cell 5] 전처리 Pipeline (Data Leakage 방지)

In [ ]:
class GroupMedianImputer(BaseEstimator, TransformerMixin):
    """
    (Pclass, Title) 그룹별 중앙값으로 target(Age) 결측치를 대체.
    - fit(): 학습 fold에서만 그룹 중앙값과 전체 중앙값을 계산
    - transform(): 저장된 값으로 대체 (검증 fold의 정보는 사용하지 않음)
    - 학습 fold에 없는 그룹 조합은 전체 중앙값으로 보완
    """

    def __init__(self, target: str = "Age", groups: tuple = ("Pclass", "Title")):
        self.target = target
        self.groups = groups

    def fit(self, X: pd.DataFrame, y=None):
        self.group_medians_ = X.groupby(list(self.groups))[self.target].median()
        self.global_median_ = X[self.target].median()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        keys = pd.MultiIndex.from_frame(X[list(self.groups)])
        fill = pd.Series(self.group_medians_.reindex(keys).values, index=X.index)
        X[self.target] = X[self.target].fillna(fill).fillna(self.global_median_)
        return X


def build_preprocessor() -> Pipeline:
    """GroupMedianImputer -> ColumnTransformer(수치형 스케일링 + 범주형 원-핫 인코딩)."""
    column_tf = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),        # 안전장치 (Age는 이미 대체됨)
            ("scale", StandardScaler()),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),  # Embarked 결측 2건 -> 최빈값
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ]), CAT_COLS),
        ("pass", "passthrough", PASS_COLS),
    ])
    return Pipeline([
        ("age_impute", GroupMedianImputer("Age", ("Pclass", "Title"))),
        ("column_tf", column_tf),
    ])


def build_pipeline(model: BaseEstimator) -> Pipeline:
    """전처리 + 모델을 하나의 Pipeline으로 캡슐화 -> fold마다 전체가 재학습됨."""
    return Pipeline([("prep", build_preprocessor()), ("model", model)])

### [Cell 6] 모델 레지스트리: (모델 클래스, 고정 파라미터, Optuna 탐색 공간)

In [ ]:
def _knn(t: optuna.Trial) -> dict:
    return {
        "n_neighbors": t.suggest_int("n_neighbors", 3, 31),
        "weights": t.suggest_categorical("weights", ["uniform", "distance"]),
        "p": t.suggest_int("p", 1, 2),  # 1 = Manhattan, 2 = Euclidean
    }


def _logreg(t: optuna.Trial) -> dict:
    return {
        "C": t.suggest_float("C", 1e-3, 1e2, log=True),  # 정규화 강도의 역수
        "penalty": t.suggest_categorical("penalty", ["l1", "l2"]),
    }


def _nb(t: optuna.Trial) -> dict:
    return {"var_smoothing": t.suggest_float("var_smoothing", 1e-11, 1e-5, log=True)}


def _tree(t: optuna.Trial) -> dict:
    return {
        "criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth": t.suggest_int("max_depth", 2, 12),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
    }


def _rf(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 15),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
        "max_features": t.suggest_categorical("max_features", ["sqrt", "log2"]),
    }


def _xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 8),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 7),
        "gamma": t.suggest_float("gamma", 0.0, 0.5),
    }


MODEL_REGISTRY: dict[str, tuple[type, dict, Callable]] = {
    "KNN": (KNeighborsClassifier, {}, _knn),
    "Logistic Regression": (
        LogisticRegression,
        {"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_STATE},  # liblinear: L1/L2 모두 지원
        _logreg,
    ),
    "Naive Bayes": (GaussianNB, {}, _nb),
    "Decision Tree": (DecisionTreeClassifier, {"random_state": RANDOM_STATE}, _tree),
    "Random Forest": (RandomForestClassifier, {"random_state": RANDOM_STATE}, _rf),
    "XGBoost": (
        XGBClassifier,
        {"random_state": RANDOM_STATE, "eval_metric": "logloss"},
        _xgb,
    ),
}


def make_model(name: str, params: dict | None = None) -> BaseEstimator:
    """레지스트리의 고정 파라미터 + (선택) 튜닝 파라미터로 모델 생성."""
    cls, fixed, _ = MODEL_REGISTRY[name]
    return cls(**fixed, **(params or {}))

### [Cell 7] 평가 함수 (Stratified 5-Fold, 7개 지표)

In [ ]:
def evaluate_cv(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, cv) -> dict:
    """
    fold마다 Pipeline을 clone하여 학습 fold로만 fit -> 검증 fold로 평가.
    반환값: 7개 지표의 fold 평균 (FN+FP는 fold당 평균 오분류 건수).
    """
    scores = {m: [] for m in METRIC_COLS}
    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(pipeline).fit(X_tr, y_tr)
        y_pred = model.predict(X_te)                 # 임계값 0.5 기준 예측
        y_prob = model.predict_proba(X_te)[:, 1]     # 순위 기반 지표(PR_AUC, AUROC)용

        tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
        scores["FN+FP"].append(fn + fp)
        scores["FPR"].append(fp / (fp + tn) if (fp + tn) else 0.0)
        scores["Recall"].append(recall_score(y_te, y_pred, zero_division=0))
        scores["Precision"].append(precision_score(y_te, y_pred, zero_division=0))
        scores["F1-Score"].append(f1_score(y_te, y_pred, zero_division=0))
        scores["PR_AUC"].append(average_precision_score(y_te, y_prob))
        scores["AUROC"].append(roc_auc_score(y_te, y_prob))

    return {m: float(np.mean(v)) for m, v in scores.items()}


def evaluate_models(models: dict[str, BaseEstimator], X, y, cv) -> pd.DataFrame:
    """여러 모델을 동일한 fold에서 평가하여 [Model | 7개 지표] 표 반환."""
    rows = [{"Model": name, **evaluate_cv(build_pipeline(m), X, y, cv)}
            for name, m in models.items()]
    return pd.DataFrame(rows, columns=["Model", *METRIC_COLS]).round(4)

### [Cell 8] Optuna 하이퍼파라미터 튜닝

In [ ]:
def tune_model(name: str, X, y, cv, n_trials: int = N_TRIALS,
               scoring: str = TUNING_METRIC) -> optuna.Study:
    """TPE Sampler(seed 고정)로 평균 CV F1을 최대화. trial마다 전처리 포함 Pipeline 전체를 재학습."""
    _, _, space_fn = MODEL_REGISTRY[name]

    def objective(trial: optuna.Trial) -> float:
        pipeline = build_pipeline(make_model(name, space_fn(trial)))
        # n_jobs=1: 병렬 실행에 따른 비결정성 방지
        return cross_val_score(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=1).mean()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        study_name=name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study


def tune_all_models(X, y, cv) -> dict[str, optuna.Study]:
    studies = {}
    for name in MODEL_REGISTRY:
        print(f"\n▶ {name} 튜닝 중 ({N_TRIALS} trials, 기준: {TUNING_METRIC})")
        studies[name] = tune_model(name, X, y, cv)
    return studies


def summarize_studies(studies: dict[str, optuna.Study]) -> pd.DataFrame:
    """모델별 Best CV F1과 Best Params 요약 (F1 내림차순)."""
    return pd.DataFrame([
        {"Model": name,
         "Best CV F1": round(s.best_value, 4),
         # 유효숫자 4자리 (예: var_smoothing=1.767e-09가 0.0으로 표시되지 않도록)
         "Best Params": {k: (float(f"{v:.4g}") if isinstance(v, float) else v)
                         for k, v in s.best_params.items()}}
        for name, s in studies.items()
    ]).sort_values("Best CV F1", ascending=False, ignore_index=True)

### [Cell 9] 출력 헬퍼 (Colab에서 최고 성능 셀 강조)

In [ ]:
def show_metrics(df: pd.DataFrame, title: str) -> None:
    """지표별 최고값을 굵게 + 배경색으로 강조 (FN+FP, FPR은 낮을수록 우수)."""
    print(f"\n[{title}]")
    if not IN_NOTEBOOK:
        print(df.to_string(index=False))
        return
    higher = [c for c in METRIC_COLS if c not in LOWER_IS_BETTER]
    lower = [c for c in METRIC_COLS if c in LOWER_IS_BETTER]
    best = "font-weight: bold; background-color: rgba(76, 175, 80, 0.25)"
    styled = (
        df.style
        .format({c: "{:.1f}" if c == "FN+FP" else "{:.4f}" for c in METRIC_COLS})
        .highlight_max(subset=higher, props=best)
        .highlight_min(subset=lower, props=best)
        .hide(axis="index")
    )
    display(styled)


def delta_table(baseline: pd.DataFrame, tuned: pd.DataFrame) -> pd.DataFrame:
    """Tuned - Baseline 변화량 (FN+FP, FPR은 음수일수록 개선)."""
    b = baseline.set_index("Model")[METRIC_COLS]
    t = tuned.set_index("Model")[METRIC_COLS]
    return (t - b).round(4).reset_index()

### [Cell 10] 실행

In [ ]:
# (0) 데이터 확인
X, y = load_and_clean()
print(f"Data shape: {X.shape}, 생존율: {y.mean():.2%}")
print(summarize(X).to_string(index=False))

# 모든 단계(Baseline, Optuna, Tuned)에서 동일한 fold 분할 사용 -> 공정한 비교
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# (1) Baseline: 기본 하이퍼파라미터
baseline_df = evaluate_models({n: make_model(n) for n in MODEL_REGISTRY}, X, y, cv)
show_metrics(baseline_df, "Baseline 5-Fold 성능")

# (2) Optuna 튜닝: 6개 모델
studies = tune_all_models(X, y, cv)

# (3) 모델별 최적 하이퍼파라미터
print("\n" + "=" * 70)
for name, s in studies.items():
    print(f"[{name}] best {TUNING_METRIC}: {s.best_value:.4f}")
    print(f"    best params: {s.best_params}")
print("=" * 70)
best_summary = summarize_studies(studies)
if IN_NOTEBOOK:
    display(best_summary)
else:
    print(best_summary.to_string(index=False))

# (4) Tuned: 최적 파라미터로 동일 fold 재평가
tuned_df = evaluate_models(
    {n: make_model(n, s.best_params) for n, s in studies.items()}, X, y, cv
)
show_metrics(tuned_df, "Tuned 5-Fold 성능")

# (5) Baseline 대비 변화량
print("\n[Tuned − Baseline 변화량]  (FN+FP, FPR은 음수 = 개선)")
print(delta_table(baseline_df, tuned_df).to_string(index=False))